# 03 · Image generation — from brief to campaign poster

Marketing needs a seasonal sale poster and doesn't have a designer free this week. Can `MAI-Image-2.5-Pro` turn a plain-English brief into something usable — and can a model judge its own output well enough to catch a bad brief before a human has to?

**Learning objectives**
- Generate a campaign poster from a marketing brief
- Score it yourself against a rubric — the fast, manual way
- Automate that scoring with `gpt-5.4` as an LLM judge, feeding it the image + rubric
- Iterate the brief and confirm the judge's score actually improves

`~20 minutes`


## 1 · Set up and a helper to generate images

Same `.env` as every other lab. MAI-Image uses a **dedicated endpoint**, not the OpenAI chat route — `generate_image(prompt)` wraps that call and returns a base64 PNG.


In [ ]:
import base64
import json
import os
from pathlib import Path
from urllib.parse import urlparse

import requests
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from IPython.display import HTML, display


def find_agent_builder() -> Path:
    """Locate foundry/agent-builder from anywhere in the tree (repo root or labs/more)."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "foundry" / "agent-builder" / "src").is_dir():
            return base / "foundry" / "agent-builder"
        if base.name == "agent-builder" and (base / "src").is_dir():
            return base
    raise FileNotFoundError("Run labs/core/00-validate-setup.ipynb first — src/.env not found.")


AB = find_agent_builder()
load_dotenv(AB / "src" / ".env")
project_client = AIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential(),
)
openai_client = project_client.get_openai_client()

# MAI lives on a dedicated endpoint, not the OpenAI chat route.
resource = urlparse(os.environ["FOUNDRY_PROJECT_ENDPOINT"])
mai_url = f"{resource.scheme}://{resource.netloc}/mai/v1/images/generations"
mai_token = DefaultAzureCredential().get_token("https://cognitiveservices.azure.com/.default").token


def generate_image(prompt: str) -> str:
    """Call MAI-Image-2.5-Pro and return the generated image as a base64 PNG string."""
    r = requests.post(
        mai_url,
        headers={"Content-Type": "application/json", "Authorization": f"Bearer {mai_token}"},
        json={"model": "MAI-Image-2.5-Pro", "prompt": prompt, "width": 1024, "height": 1024},
        timeout=120,
    )
    r.raise_for_status()
    return r.json()["data"][0]["b64_json"]


def show_image(b64: str, caption: str = "") -> None:
    display(HTML(f'<figure style="margin:0"><img src="data:image/png;base64,{b64}" width="420"/>'
                  f'<figcaption>{caption}</figcaption></figure>'))


print("Ready. `generate_image(prompt)` returns a base64 PNG; `show_image(b64, caption)` displays it.")


## 2 · Generate a poster from a plain brief

Marketing's brief, verbatim: *"Fall hiking sale poster, family with backpacks in an autumn forest, warm and inviting."*

> ❓ **Is a one-line brief enough to get something poster-worthy on the first try?**


In [ ]:
BRIEF_V1 = "Fall hiking sale poster, family with backpacks in an autumn forest, warm and inviting."

poster_v1 = generate_image(BRIEF_V1)
show_image(poster_v1, "Poster v1 — from the one-line brief")


## 3 · Score it yourself, against a rubric

Before automating anything, look at the poster above and score it 1–5 on each of these — this is the same checklist a marketing reviewer would use:

| Criterion | What to look for |
|---|---|
| **Composition** | Is there empty space for a headline and price, or is the frame already busy? |
| **Brand fit** | Does it feel like an outdoor gear brand, not a stock-photo generic? |
| **Realism** | Any obvious rendering artifacts (extra limbs, warped gear, garbled text)? |

> ❓ **Your call:** score poster v1. Where did it fall short? Keep that in mind for section 5.


## 4 · Automate the score — an LLM as judge

Manually scoring one poster is fine. Scoring fifty campaign variants isn't. We send the same rubric plus the image to `gpt-5.4` (a vision-capable model) and ask for structured scores instead of prose.

> ❓ **Does the model's score line up with your own read of poster v1?**


In [ ]:
JUDGE_PROMPT = """
Score this campaign poster 1-5 on each criterion below. Return ONLY a JSON object
with keys: composition, brand_fit, realism (each an int 1-5), and notes (one
short sentence on the biggest weakness, or "none" if there isn't one).

- composition: is there clear empty space for a headline and price?
- brand_fit: does it feel like an outdoor gear brand, not generic stock art?
- realism: any rendering artifacts (extra limbs, warped gear, garbled text)?
"""


def judge_poster(model: str, image_b64: str, prompt: str = JUDGE_PROMPT) -> dict:
    """Ask a vision model to score a poster and return the parsed JSON scores."""
    resp = openai_client.chat.completions.create(
        model=model,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
            ],
        }],
    )
    answer = resp.choices[0].message.content
    return json.loads(answer[answer.find("{"): answer.rfind("}") + 1])


scores_v1 = judge_poster("gpt-5.4", poster_v1)
print("Judge's scores for poster v1:")
print(json.dumps(scores_v1, indent=2))


## 5 · Iterate the brief, re-judge, and compare

Take whatever the judge flagged in `notes` and write a sharper brief — explicit about composition and what to avoid. If the automated judge is trustworthy, its score should move in the direction you'd expect.

> ❓ **Does a more specific brief actually raise the score, or just change the picture?**


In [ ]:
BRIEF_V2 = (
    "Fall hiking sale poster for an outdoor gear brand. A family of three wearing "
    "hiking backpacks on a trail in an autumn forest, warm golden-hour light. "
    "Leave the top third of the frame as clear open sky for a headline and price "
    "overlay. No text, no logos, no signage in the image itself. Photorealistic, "
    "no visual artifacts."
)

poster_v2 = generate_image(BRIEF_V2)
show_image(poster_v2, "Poster v2 — from the sharpened brief")

scores_v2 = judge_poster("gpt-5.4", poster_v2)
print("Judge's scores for poster v2:")
print(json.dumps(scores_v2, indent=2))

print("\n--- v1 vs v2 ---")
for key in ("composition", "brand_fit", "realism"):
    print(f"  {key:12} v1={scores_v1[key]}  v2={scores_v2[key]}")


## 6 · When to automate the review

| Situation | Approach |
|---|---|
| One-off poster, low volume | Manual review — a person looks at it, done |
| Dozens of campaign variants, or an ongoing pipeline | LLM-as-judge — same rubric, applied consistently and fast |
| High-stakes / brand-critical asset | Both — let the judge triage, have a human sign off before it ships |

**Rule of thumb:** a vague brief produces a vague image — be as explicit about layout and exclusions as you would with a human designer. The LLM judge is only as useful as the rubric you give it.


## 🧭 Summary — so, can a model brief-to-poster and self-review?

Yes: `MAI-Image-2.5-Pro` generated a usable poster, and `gpt-5.4` scored it against the same rubric a human would — and the score moved when the brief got sharper.

### Try it yourself
- Ask the judge to score against a rubric criterion you care about (e.g., "does this look like winter, not fall?").
- Try a deliberately bad brief and confirm the judge's score drops accordingly.

### Next
➡️ **[04 · Model Router](04-model-router.ipynb)** — see how one router deployment picks a different model per request, automatically.
